In [1]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]
print(os.environ["HADOOP_HOME"])

C:\hadoop


In [2]:
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, os, json

builder = (SparkSession.builder
    .appName("EP3-Delta-Lake")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} con Delta Lake listo.")

Spark 3.5.3 con Delta Lake listo.


In [ ]:
RUTA_SILVER_PARQUET = "outputs/silver/dataset_silver.parquet" 

df_silver = spark.read.parquet(RUTA_SILVER_PARQUET)
print(f"Filas: {df_silver.count():,}")
df_silver.printSchema()

Filas: 7,728,329
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance_mi: double (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: string (nullable = true)
 |-- Temperature_F: double (nullable = true)
 |-- Humidity_pct: double (nullable = true)
 |-- Pressure_in: double (nullable = true)
 |-- Visibility_mi: double (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed_mph: double (nullable = true)
 |-- Precipitation_in: double (nullable = true)
 |-- Weather_C

In [4]:
RUTA_SILVER_DELTA = "outputs/silver_delta"

t0 = time.time()
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("compression", "zstd")
    .save(RUTA_SILVER_DELTA))
print(f"Delta escrito en {time.time()-t0:.2f}s")

log_files = sorted(os.listdir(f"{RUTA_SILVER_DELTA}/_delta_log/"))
print(f"\nDelta Log contiene {len(log_files)} archivo(s):")
for f in log_files:
    print(" ", f)

Delta escrito en 42.72s

Delta Log contiene 3 archivo(s):
  .00000000000000000000.json.crc
  00000000000000000000.json
  _commits


In [5]:
with open(f"{RUTA_SILVER_DELTA}/_delta_log/00000000000000000000.json") as f:
    for line in f.readlines()[:5]:
        print(json.dumps(json.loads(line), indent=2)[:600])
        print("-" * 40)

{
  "commitInfo": {
    "timestamp": 1793936031838,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "Overwrite",
      "partitionBy": "[]"
    },
    "isolationLevel": "Serializable",
    "isBlindAppend": false,
    "operationMetrics": {
      "numFiles": "13",
      "numOutputRows": "7728329",
      "numOutputBytes": "406618102"
    },
    "engineInfo": "Apache-Spark/3.5.3 Delta-Lake/3.2.1",
    "txnId": "7432cb92-f62a-4bef-9cb6-82fe5497ad87"
  }
}
----------------------------------------
{
  "metaData": {
    "id": "0d7222a7-042e-4927-9877-6d82871e92b7",
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"ID\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"Source\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"Severity\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}},{\"name\":\"Start_Time\",\"type\":\"timestamp\",\"nullable\":tr

### Análisis del Delta Log
- La entrada 0 registra: `protocol` (versiones mínimas de lector/escritor), `metaData` (esquema, formato, fecha de creación) y una acción `add` por cada archivo Parquet escrito.
- Campos de `add`: `path`, `size`, `modificationTime`, `dataChange` y `stats`.
- `stats` guarda por archivo: número de filas, mínimos, máximos y nulos por columna. Delta los usa para saltarse archivos que no cumplen un filtro (data skipping) sin abrirlos.

In [6]:
tabla_silver = DeltaTable.forPath(spark, RUTA_SILVER_DELTA)
print("Historial inicial:")
tabla_silver.history().select("version","timestamp","operation").show(truncate=False)

df = tabla_silver.toDF()
print("Máximo Distance_mi:", df.agg(F.max("Distance_mi")).collect()[0][0])
for corte in [50, 100, 200, 500]:
    print(f"Distance_mi > {corte}: {df.filter(F.col('Distance_mi') > corte).count():,}")

Historial inicial:
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-11-05 20:33:52.096|WRITE    |
+-------+-----------------------+---------+

Máximo Distance_mi: 441.75
Distance_mi > 50: 715
Distance_mi > 100: 86
Distance_mi > 200: 9
Distance_mi > 500: 0


In [7]:
COL_NUMERICA = "Distance_mi"
VALOR_CORTE = 100   # AJUSTAR

n_antes = df.filter(F.col(COL_NUMERICA) > VALOR_CORTE).count()
print(f"Registros con {COL_NUMERICA} > {VALOR_CORTE}: {n_antes:,}")

tabla_silver.update(
    condition=F.col(COL_NUMERICA) > VALOR_CORTE,
    set={COL_NUMERICA: F.lit(None).cast("double")}
)

n_despues = tabla_silver.toDF().filter(F.col(COL_NUMERICA) > VALOR_CORTE).count()
print(f"Después del UPDATE: {n_despues:,}")

Registros con Distance_mi > 100: 86
Después del UPDATE: 0


In [11]:
tabla_silver.history().select("version","timestamp","operation").show(truncate=False)

+-------+-----------------------+-----------+
|version|timestamp              |operation  |
+-------+-----------------------+-----------+
|2      |2026-11-05 20:40:52.889|ADD COLUMNS|
|1      |2026-11-05 20:38:11.503|UPDATE     |
|0      |2026-11-05 20:33:52.096|WRITE      |
+-------+-----------------------+-----------+



In [13]:
tabla_silver = DeltaTable.forPath(spark, RUTA_SILVER_DELTA)

df_correcciones = (tabla_silver.toDF()
    .sample(fraction=0.001, seed=42)
    .limit(50)
    .withColumn("Source", F.lit("CORREGIDO"))
    .withColumn("flag_corregido", F.lit(True)))
print(f"Correcciones a aplicar: {df_correcciones.count()}")

(tabla_silver.alias("t")
    .merge(df_correcciones.alias("c"), "t.ID = c.ID")
    .whenMatchedUpdateAll()
    .execute())

tabla_silver.history().select("version","timestamp","operation","operationMetrics").show(truncate=False)

Correcciones a aplicar: 50
+-------+-----------------------+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation  |operationMetrics                                                                                                                                                                                                                   

In [14]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(RUTA_SILVER_DELTA)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(RUTA_SILVER_DELTA)
df_actual = spark.read.format("delta").load(RUTA_SILVER_DELTA)

print(f"Filas v0: {df_v0.count():,} | v1: {df_v1.count():,} | actual: {df_actual.count():,}")

for nombre, d in [("v0", df_v0), ("v1", df_v1), ("actual", df_actual)]:
    nulos = d.filter(F.col("Distance_mi").isNull()).count()
    mx = d.agg(F.max("Distance_mi")).collect()[0][0]
    print(f"{nombre}: nulos Distance_mi = {nulos:,} | máximo = {mx}")

print("Registros corregidos (actual):",
      df_actual.filter(F.col("flag_corregido") == True).count())
print("¿v0 tiene flag_corregido?", "flag_corregido" in df_v0.columns)

Filas v0: 7,728,329 | v1: 7,728,329 | actual: 7,728,329
v0: nulos Distance_mi = 0 | máximo = 441.75
v1: nulos Distance_mi = 86 | máximo = 99.952
actual: nulos Distance_mi = 86 | máximo = 99.952
Registros corregidos (actual): 50
¿v0 tiene flag_corregido? False


### ¿Para qué sirve el time travel en mi dataset?
Escenario: el UPDATE convirtió en nulo `Distance_mi` en 86 accidentes con más de 100 millas. Si después se determina que alguno era un valor real (por ejemplo, un cierre largo de carretera), puedo leer la versión 0 y recuperar el valor original sin restaurar un respaldo. También permite auditar qué cambió entre versiones y reproducir un análisis exacto con los datos de una fecha dada.

In [15]:
tabla_silver = DeltaTable.forPath(spark, RUTA_SILVER_DELTA)

def estado():
    r = tabla_silver.detail().select("numFiles","sizeInBytes").collect()[0]
    return r.numFiles, r.sizeInBytes/1024/1024

nf, mb = estado()
print(f"Antes de OPTIMIZE: {nf} archivos, {mb:.1f} MB")

t0 = time.time()
tabla_silver.optimize().executeCompaction()
print(f"OPTIMIZE tardó {time.time()-t0:.1f}s")

nf, mb = estado()
print(f"Después de OPTIMIZE: {nf} archivos, {mb:.1f} MB")

tabla_silver.history().select("version","operation","operationMetrics").show(1, truncate=False)

Antes de OPTIMIZE: 13 archivos, 518.8 MB
OPTIMIZE tardó 65.2s
Después de OPTIMIZE: 1 archivos, 537.1 MB
+-------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                                                       |
+-------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|4      |OPTIMIZE |{numRemovedFiles -> 13, n

In [20]:
def mb_disco(path):
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(path) for f in fs) / 1024 / 1024

print(f"En disco antes de VACUUM: {mb_disco(RUTA_SILVER_DELTA):.1f} MB")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
tabla_silver.vacuum(0)
print(f"En disco después de VACUUM: {mb_disco(RUTA_SILVER_DELTA):.1f} MB")

try:
    d0 = spark.read.format("delta").option("versionAsOf", 0).load(RUTA_SILVER_DELTA)
    filas = d0.select("ID", "Street", "Distance_mi").limit(5).collect()
    print(filas)
    print("v0 todavía accesible")
except Exception as e:
    print("v0 ya no se puede leer:", str(e)[:300])

En disco antes de VACUUM: 541.3 MB
En disco después de VACUUM: 541.3 MB
v0 ya no se puede leer: An error occurred while calling o330.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 508.0 failed 1 times, most recent failure: Lost task 0.0 in stage 508.0 (TID 33392) (LaptopAle executor driver): org.apache.spark.SparkFileNotFoundException: Fil


- OPTIMIZE compactó 13 archivos en 1 (65.2 s). Menos archivos significa menos aperturas y menos metadatos por query, pero un solo archivo de 537 MB reduce el paralelismo de lectura. El tamaño pasó de 518.8 a 537.1 MB porque UPDATE y MERGE reescribieron los archivos sin ZSTD.
- VACUUM(0) liberó espacio en disco (1497.4 MB a 541.3 MB) pero elimina el time travel a versiones anteriores. En producción se usa retentionHours >= 168 para no borrar archivos que un lector lento aún necesita ni perder el historial reciente.

In [18]:
for r, d, f in os.walk("outputs/gold"):
    print(r, d[:5], f[:5])

outputs/gold ['agregacion_principal.parquet', 'analisis_ventana.parquet', 'consulta_sql.parquet'] []
outputs/gold\agregacion_principal.parquet [] ['.part-00000-9dc2b5d2-851d-4c6f-a171-cc9c757166ea-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-9dc2b5d2-851d-4c6f-a171-cc9c757166ea-c000.snappy.parquet', '_SUCCESS']
outputs/gold\analisis_ventana.parquet [] ['.part-00000-d74e2d08-e861-496c-9086-733afde5e112-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-d74e2d08-e861-496c-9086-733afde5e112-c000.snappy.parquet', '_SUCCESS']
outputs/gold\consulta_sql.parquet [] ['.part-00000-7a7f2cc8-a81e-40a6-936d-0520f4a75560-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-7a7f2cc8-a81e-40a6-936d-0520f4a75560-c000.snappy.parquet', '_SUCCESS']


In [19]:
RUTA_GOLD_PARQUET = "outputs/gold"
RUTA_GOLD_DELTA = "outputs/gold_delta"
tablas_gold = ["agregacion_principal", "analisis_ventana", "consulta_sql"]

for nombre in tablas_gold:
    origen = f"{RUTA_GOLD_PARQUET}/{nombre}.parquet"
    destino = f"{RUTA_GOLD_DELTA}/{nombre}"
    d = spark.read.parquet(origen)
    (d.write.format("delta").mode("overwrite")
        .option("compression", "zstd")
        .save(destino))
    n = spark.read.format("delta").load(destino).count()
    log = sorted(os.listdir(f"{destino}/_delta_log"))
    print(f"{nombre}: {n:,} filas | columnas: {len(d.columns)} | log: {log[:2]}")

agregacion_principal: 49 filas | columnas: 7 | log: ['.00000000000000000000.json.crc', '00000000000000000000.json']
analisis_ventana: 388 filas | columnas: 7 | log: ['.00000000000000000000.json.crc', '00000000000000000000.json']
consulta_sql: 240 filas | columnas: 4 | log: ['.00000000000000000000.json.crc', '00000000000000000000.json']
